# Medical Diagnosis Assistant — RAG with LangChain + OpenAI


## Project Overview
This notebook implements a **Retrieval-Augmented Generation (RAG)** system that assists with
diagnosis suggestions by combining two knowledge sources:

1. **Patient Medical Histories** — built from `healthcare_patients.csv` and
   `healthcare_encounters.csv` (demographics + chronological encounter notes/diagnoses).
2. **Medical Knowledge Base** — a curated reference of the ICD-10 condition codes present in
   the dataset (description, typical symptoms, risk factors, standard management), used to
   ground the model's reasoning in clinical context rather than free association.

## Architecture

```
                 ┌─────────────────────────┐
                 │   User Query             │
                 │  (patient_id or symptom  │
                 │   description)           │
                 └────────────┬─────────────┘
                              │
              ┌───────────────┴───────────────┐
              │                                │
   ┌──────────▼──────────┐          ┌──────────▼───────────┐
   │ Patient History      │          │ Medical Knowledge     │
   │ FAISS Vector Store    │          │ FAISS Vector Store    │
   │ (sentence-transformers)│         │ (sentence-transformers)│
   └──────────┬──────────┘          └──────────┬───────────┘
              │        Retrieved context         │
              └───────────────┬───────────────────┘
                              │
                    ┌─────────▼─────────┐
                    │  LangChain Prompt   │
                    │  + OpenAI GPT-4o-mini    │
                    │  (generation)       │
                    └─────────┬─────────┘
                              │
                    ┌─────────▼─────────┐
                    │ Diagnosis Suggestion│
                    │ + supporting evidence│
                    └────────────────────┘
```




## 1. Environment Setup

Run this notebook inside the virtual environment described in `README.md`

Your OpenAI API key must be set in a `.env` file in the project root:



In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the project root

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, (
    "OPENAI_API_KEY not found. Create a .env file in the project root "
    "with: OPENAI_API_KEY=sk-...your-key..."
)
print("✅ OpenAI API key loaded.")


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

pd.set_option("display.max_colwidth", 120)
print("✅ Libraries imported.")


## 2. Load the Dataset

- `healthcare_patients.csv` — 400 patients: `patient_id, age, sex, bmi, smoker, diabetic`
- `healthcare_encounters.csv` — 1,500 encounters: `encounter_id, patient_id, date, dx_code, note`


In [ ]:
patients_df = pd.read_csv("data/healthcare_patients.csv")
encounters_df = pd.read_csv("data/healthcare_encounters.csv")
encounters_df["date"] = pd.to_datetime(encounters_df["date"])

print("Patients:", patients_df.shape)
print("Encounters:", encounters_df.shape)
display(patients_df.head())
display(encounters_df.head())


In [ ]:
# Quick sanity check on the diagnosis codes present in the data
print(encounters_df["dx_code"].value_counts())


## 3. Build the Medical Knowledge Base



In [ ]:
# Curated ICD-10 reference for the codes present in this dataset.
MEDICAL_KNOWLEDGE = {
    "E11": {
        "name": "Type 2 Diabetes Mellitus",
        "symptoms": "increased thirst, frequent urination, fatigue, blurred vision, slow-healing sores",
        "risk_factors": "obesity/high BMI, sedentary lifestyle, family history, age over 45",
        "management": "metformin or other glucose-lowering agents, diet and exercise, regular A1c monitoring",
    },
    "E78": {
        "name": "Disorders of Lipoprotein Metabolism (Hyperlipidemia)",
        "symptoms": "usually asymptomatic; found via lipid panel; may contribute to atherosclerosis",
        "risk_factors": "high-fat diet, obesity, smoking, diabetes, family history",
        "management": "statins, lifestyle modification, periodic lipid panel monitoring",
    },
    "F41": {
        "name": "Anxiety Disorder (Other)",
        "symptoms": "excessive worry, restlessness, muscle tension, difficulty concentrating, sleep disturbance",
        "risk_factors": "chronic stress, family history, comorbid depression, substance use",
        "management": "cognitive behavioral therapy, SSRIs/SNRIs, stress-reduction techniques",
    },
    "G47": {
        "name": "Sleep Disorder",
        "symptoms": "difficulty falling/staying asleep, daytime fatigue, non-restorative sleep",
        "risk_factors": "stress, poor sleep hygiene, obesity (sleep apnea), shift work",
        "management": "sleep hygiene counseling, CPAP if apnea, short-term sleep aids if indicated",
    },
    "I10": {
        "name": "Essential (Primary) Hypertension",
        "symptoms": "often asymptomatic; occasionally headache, dizziness",
        "risk_factors": "high sodium intake, obesity, smoking, family history, age",
        "management": "ACE inhibitors/ARBs, diuretics, lifestyle changes, regular BP monitoring",
    },
    "I25": {
        "name": "Chronic Ischemic Heart Disease",
        "symptoms": "chest pain/pressure, fatigue, shortness of breath on exertion",
        "risk_factors": "hyperlipidemia, hypertension, smoking, diabetes, family history",
        "management": "statins, antiplatelet therapy, beta-blockers, lifestyle modification, cardiology follow-up",
    },
    "J45": {
        "name": "Asthma",
        "symptoms": "wheezing, shortness of breath, chest tightness, cough (often nocturnal)",
        "risk_factors": "allergies, smoking/secondhand smoke exposure, family history, respiratory infections",
        "management": "inhaled corticosteroids, bronchodilators (rescue inhaler), trigger avoidance",
    },
    "K21": {
        "name": "Gastro-Esophageal Reflux Disease (GERD)",
        "symptoms": "heartburn, regurgitation, chest discomfort, worse after meals or lying down",
        "risk_factors": "obesity, smoking, hiatal hernia, certain foods/alcohol",
        "management": "proton pump inhibitors/H2 blockers, dietary changes, weight management",
    },
    "M54": {
        "name": "Dorsalgia (Back Pain)",
        "symptoms": "localized or radiating back pain, stiffness, reduced range of motion",
        "risk_factors": "sedentary lifestyle, poor posture, obesity, prior injury",
        "management": "physical therapy, NSAIDs, activity modification, ergonomic changes",
    },
}

knowledge_docs = []
for code_, info in MEDICAL_KNOWLEDGE.items():
    text = (
        f"ICD-10 {code_}: {info['name']}. "
        f"Typical symptoms: {info['symptoms']}. "
        f"Risk factors: {info['risk_factors']}. "
        f"Standard management: {info['management']}."
    )
    knowledge_docs.append(
        Document(page_content=text, metadata={"dx_code": code_, "name": info["name"], "source": "medical_knowledge"})
    )

print(f"Built {len(knowledge_docs)} medical knowledge documents.")
print(knowledge_docs[0].page_content)


## 4. Embed Patient Medical Histories



In [ ]:
def build_patient_history_text(patient_row, patient_encounters):
    header = (
        f"Patient {patient_row.patient_id}: {patient_row.age}-year-old {patient_row.sex}, "
        f"BMI {patient_row.bmi}, smoker={'yes' if patient_row.smoker else 'no'}, "
        f"diabetic={'yes' if patient_row.diabetic else 'no'}."
    )
    encounter_lines = []
    for _, enc in patient_encounters.sort_values("date").iterrows():
        cond_name = MEDICAL_KNOWLEDGE.get(enc.dx_code, {}).get("name", enc.dx_code)
        encounter_lines.append(
            f"- {enc.date.date()} [{enc.dx_code} - {cond_name}]: {enc.note}"
        )
    history = "\n".join(encounter_lines) if encounter_lines else "No recorded encounters."
    return f"{header}\nEncounter history:\n{history}"


patient_history_docs = []
for row in patients_df.itertuples():
    p_encounters = encounters_df[encounters_df.patient_id == row.patient_id]
    text = build_patient_history_text(row, p_encounters)
    patient_history_docs.append(
        Document(
            page_content=text,
            metadata={
                "patient_id": row.patient_id,
                "age": row.age,
                "sex": row.sex,
                "num_encounters": len(p_encounters),
                "source": "patient_history",
            },
        )
    )

print(f"Built {len(patient_history_docs)} patient history documents.")
print(patient_history_docs[0].page_content)


## 5. Create Embeddings and FAISS Vector Stores

We use `sentence-transformers/all-MiniLM-L6-v2` (via `langchain_huggingface`) to generate
embeddings . Two separate FAISS indices are built so we can retrieve from
each knowledge source independently and control how many results come from each.



In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Building patient history vector store...")
patient_vectorstore = FAISS.from_documents(patient_history_docs, embeddings)

print("Building medical knowledge vector store...")
knowledge_vectorstore = FAISS.from_documents(knowledge_docs, embeddings)

# Persist to disk so we don't have to re-embed every run
patient_vectorstore.save_local("vector_store/patient_history")
knowledge_vectorstore.save_local("vector_store/medical_knowledge")

print("✅ Vector stores built and saved to ./vector_store/")


## 6. Build the RAG Chain (Retrieval + OpenAI Generation)

Given a query (either a `patient_id` or a free-text symptom description), we:

1. Retrieve the most relevant patient history document(s).
2. Retrieve the most relevant medical knowledge entries (based on the same query, so the
   model gets condition context even for a fresh symptom description with no patient history).
3. Feed both as context to GPT-4o-mini with a carefully constrained prompt that (a) grounds the
   response in the retrieved context, (b) requires differential-style reasoning, and
   (c) enforces the "not a real diagnosis" disclaimer.


In [ ]:
patient_retriever = patient_vectorstore.as_retriever(search_kwargs={"k": 1})
knowledge_retriever = knowledge_vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a clinical decision-support assistant used strictly for an \
educational demo. You are NOT a doctor and this is NOT real medical advice.

You will be given:
1. A patient's medical history (demographics + prior encounters), if available.
2. Relevant medical knowledge base entries retrieved for the query.

Using ONLY this retrieved context (do not invent facts not supported by it), respond with:
- A short summary of relevant findings from the patient's history (if provided).
- 1-3 most plausible diagnosis suggestions, each with the supporting evidence from the \
  context and a brief rationale.
- Any red flags that would warrant urgent in-person evaluation.
- A closing line reminding the user this is an educational tool, not medical advice, and to \
  consult a licensed clinician.
Keep the tone clinical, concise, and evidence-grounded."""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",
     "Query: {query}\n\n"
     "--- Patient History Context ---\n{patient_context}\n\n"
     "--- Medical Knowledge Context ---\n{knowledge_context}\n\n"
     "Provide your assessment.")
])

rag_chain = rag_prompt | llm
print("✅ RAG chain ready.")


In [ ]:
def get_diagnosis_suggestion(query: str, patient_id: str | None = None) -> str:
    """
    Run the full RAG pipeline.

    query       : free-text symptom description OR a patient_id lookup phrase
    patient_id  : optionally force retrieval of a specific patient's history
                  (e.g. patient_id="H10046"); otherwise the patient retriever
                  searches by the query text itself.
    """
    # --- Patient history retrieval ---
    if patient_id:
        matches = [d for d in patient_history_docs if d.metadata["patient_id"] == patient_id]
        patient_context = matches[0].page_content if matches else "No matching patient found."
    else:
        results = patient_retriever.invoke(query)
        patient_context = results[0].page_content if results else "No patient history provided."

    # --- Medical knowledge retrieval ---
    knowledge_results = knowledge_retriever.invoke(query)
    knowledge_context = "\n".join(d.page_content for d in knowledge_results)

    response = rag_chain.invoke({
        "query": query,
        "patient_context": patient_context,
        "knowledge_context": knowledge_context,
    })
    return response.content


## 7. Example Queries

### Example A — Lookup by existing patient ID


In [ ]:
result = get_diagnosis_suggestion(
    query="Summarize this patient's condition and suggest next steps.",
    patient_id="H10046",
)
print(result)


### Example B — Free-text symptom description (no patient history)

In [ ]:
result = get_diagnosis_suggestion(
    query=(
        "45-year-old patient reports intermittent chest tightness, heartburn after meals, "
        "and occasional shortness of breath climbing stairs. History of hyperlipidemia."
    )
)
print(result)


### Example C — Another existing patient, different condition profile

In [ ]:
result = get_diagnosis_suggestion(
    query="What conditions has this patient been treated for and what should the provider watch for?",
    patient_id="H10279",
)
print(result)


## 8. Evaluation Notes

**What this demonstrates:**
- Dual-corpus RAG (structured patient histories + a curated clinical knowledge base).
- Local, cost-free embeddings (sentence-transformers) paired with a hosted LLM for generation.
- A grounded prompt that constrains the model to retrieved context and enforces safety
  disclaimers.


